# Task 4 ? Hands-on: one pipeline in Power BI

## Deliverable

The target model contains one loaded table named **Shipments**, with **34 rows**, **8 contract columns**, and **578,650 kg**. The three raw-aligned staging queries remain in Power Query but have **Enable load** disabled.

## 1?2. Connect and name staging

Use **Transform Data**, not **Load**, because Power Query must record the repeatable cleaning steps before anything reaches the model. Loading immediately would expose unstandardized source structures and bypass the staging-to-storage contract.

Queries:

- `marine_stg`
- `rail_stg`
- `road_stg`
- `Shipments` (the only loaded model table)

Every transformation is visible in Query Settings ? Applied Steps. Screenshot the Applied Steps pane for one staging query as evidence.

## 3?7. Power Query design

Rail requires special handling: delete automatic **Promoted Headers** and **Changed Type**, remove the two title rows, then promote the true header row. Automatic detection was wrong because the workbook begins with a report title and a units line, not column names.

All staging queries are renamed to the exact contract before append; append matches column names, not positions. Rail tons are converted to canonical kilograms before the tons column is removed. Road blank/TOTAL/duplicate rows are removed in staging, not in the report.

### Contract columns

`ShipmentID`, `ShipDate`, `Origin`, `Destination`, `WeightKG`, `CostPKR`, `Status`, `SourceSystem`

## 8. Validation targets

The validation table records the expected model result. Column profiling must be set to **entire data set**, not only the first 1,000 rows.

In [1]:
from pathlib import Path
import pandas as pd

ROOT=Path.cwd()
if not (ROOT/"marine_shipments.csv").exists(): ROOT=Path(r"D:\University\ML Lab\Week02")
marine=pd.read_csv(ROOT/"marine_shipments.csv")
rail=pd.read_excel(ROOT/"rail_shipments.xlsx",skiprows=2)
road=pd.read_csv(ROOT/"road_shipments.csv")

marine_out=pd.DataFrame({
    "ShipmentID":marine.Shipment_ID.astype(str),
    "ShipDate":pd.to_datetime(marine.Ship_Date,format="%d/%m/%Y"),
    "Origin":marine.Origin_Port,
    "Destination":marine.Dest_Port,
    "WeightKG":marine.Weight_KG,
    "CostPKR":marine.Cost_PKR,
    "Status":marine.Status.str.strip().str.title(),
    "SourceSystem":"Marine",
})
rail_out=pd.DataFrame({
    "ShipmentID":"RL-"+rail["Consignment No"].astype(str),
    "ShipDate":pd.to_datetime(rail["Dispatch Date"]),
    "Origin":rail["From Station"],
    "Destination":rail["To Station"],
    "WeightKG":rail["Weight (Tons)"]*1000,
    "CostPKR":rail["Freight Charges (PKR)"],
    "Status":rail.Status.str.strip().str.title(),
    "SourceSystem":"Rail",
})
road_keep=road.loc[road.trip_id.notna() & road.trip_id.ne("TOTAL")].drop_duplicates("trip_id",keep="first")
road_out=pd.DataFrame({
    "ShipmentID":road_keep.trip_id.str.upper(),
    "ShipDate":pd.to_datetime(road_keep.trip_date),
    "Origin":road_keep.from_city,
    "Destination":road_keep.to_city,
    "WeightKG":road_keep.weight_kg,
    "CostPKR":road_keep.cost_pkr,
    "Status":road_keep.status.str.strip().str.title(),
    "SourceSystem":"Road",
})
Shipments=pd.concat([marine_out,rail_out,road_out],ignore_index=True)
print("rows, columns:",Shipments.shape)
print("weight kg:",Shipments.WeightKG.sum())
print("empty/error-critical fields:",Shipments[["ShipmentID","ShipDate","WeightKG"]].isna().sum().to_dict())
print("status distinct:",Shipments.Status.nunique(),Shipments.Status.unique().tolist())
print("source tonnes:",Shipments.groupby("SourceSystem").WeightKG.sum().div(1000).to_dict())
print("status counts:",Shipments.Status.value_counts().to_dict())
assert Shipments.shape==(34,8)
assert Shipments.WeightKG.sum()==578650
assert Shipments[["ShipmentID","ShipDate","WeightKG"]].isna().sum().sum()==0
assert Shipments.Status.nunique()==3
assert Shipments.groupby("SourceSystem").WeightKG.sum().div(1000).round(2).to_dict()=={"Marine":206.5,"Rail":295.25,"Road":76.9}
assert Shipments.Status.value_counts().to_dict()=={"Delivered":22,"In Transit":8,"Delayed":4}
assert Shipments.loc[Shipments.ShipmentID.eq("MR-1003"),"ShipDate"].iloc[0]==pd.Timestamp("2026-02-03")
assert Shipments.loc[Shipments.ShipmentID.eq("MR-1012"),"ShipDate"].iloc[0]==pd.Timestamp("2026-02-13")

rows, columns: (34, 8)
weight kg: 578650.0
empty/error-critical fields: {'ShipmentID': 0, 'ShipDate': 0, 'WeightKG': 0}
status distinct: 3 ['Delivered', 'In Transit', 'Delayed']
source tonnes: {'Marine': 206.5, 'Rail': 295.25, 'Road': 76.9}
status counts: {'Delivered': 22, 'In Transit': 8, 'Delayed': 4}


### Required Power BI visuals

- Card: `SUM(Shipments[WeightKG])` = **578,650**.
- Bar chart: `SourceSystem` with `SUM(WeightKG) / 1000`: Marine **206.5 t**, Rail **295.25 t**, Road **76.9 t**.
- Status table/bar: Delivered **22**, In Transit **8**, Delayed **4**.
- Power Query profiling: 34 rows, 8 columns; no Error or Empty for `ShipmentID`, `ShipDate`, or `WeightKG`; exactly 3 status values.

## 9. M code ? `rail_stg`

Paste the following in the Advanced Editor after connecting to `rail_shipments.xlsx`. The comments annotate three important lines.

```powerquery
let
    Source = Excel.Workbook(File.Contents("D:\University\ML Lab\Week02\rail_shipments.xlsx"), null, true),
    Sheet = Source{[Item="Sheet1", Kind="Sheet"]}[Data],
    RemovedTopRows = Table.Skip(Sheet, 2),
    PromotedHeaders = Table.PromoteHeaders(RemovedTopRows, [PromoteAllScalars=true]),
    ChangedTypes = Table.TransformColumnTypes(PromotedHeaders, {
        {"Consignment No", Int64.Type}, {"Dispatch Date", type date},
        {"From Station", type text}, {"To Station", type text},
        {"Weight (Tons)", type number}, {"Freight Charges (PKR)", type number}, {"Status", type text}
    }),
    AddedShipmentID = Table.AddColumn(ChangedTypes, "ShipmentID", each "RL-" & Text.From([Consignment No]), type text),
    AddedWeightKG = Table.AddColumn(AddedShipmentID, "WeightKG", each [Weight (Tons)] * 1000, type number),
    RemovedTons = Table.RemoveColumns(AddedWeightKG, {"Consignment No", "Weight (Tons)"}),
    Renamed = Table.RenameColumns(RemovedTons, {
        {"Dispatch Date", "ShipDate"}, {"From Station", "Origin"},
        {"To Station", "Destination"}, {"Freight Charges (PKR)", "CostPKR"}
    }),
    CleanStatus = Table.TransformColumns(Renamed, {{"Status", each Text.Proper(Text.Trim(_)), type text}}),
    AddedSourceSystem = Table.AddColumn(CleanStatus, "SourceSystem", each "Rail", type text),
    Contract = Table.SelectColumns(AddedSourceSystem, {"ShipmentID", "ShipDate", "Origin", "Destination", "WeightKG", "CostPKR", "Status", "SourceSystem"})
in
    Contract
```


### Three M-code annotations

1. `Table.Skip(Sheet, 2)` removes the two report title/unit rows before headers are promoted; otherwise Power Query would use a title as a field name.
2. `Table.AddColumn(... [Weight (Tons)] * 1000 ...)` converts the source unit into the conformed `WeightKG` contract; the conversion occurs in staging, not in the dashboard.
3. `Table.SelectColumns(...)` enforces the exact eight-column contract so Append matches the three queries by name.

## Tier and load explanation

Disabling load on `marine_stg`, `rail_stg`, and `road_stg` keeps raw-aligned transformations in the **staging tier** without exposing them as separate model tables. `Shipments` is the integrated/storage contract and the presentation layer reads one trusted table; if staging load remained enabled, a business user would see four tables, duplicate/uncanonical records, and could accidentally build visuals from an intermediate query instead of the validated table.

## Bonus ? Folder connector

A Folder connector builds a parameterized sample-file query, transformation function, and combined output query for all files in the folder. New files can be included on refresh without creating a new manual query for each file. The folder is only as clean as its worst file because the same transformation function is applied to every file, so one malformed header, wrong unit, or junk row can propagate into the combined result unless validation gates reject it.

## Evidence checklist

- Transform Data used instead of direct Load.
- Three staging queries named with `_stg`.
- Rail title rows removed before header promotion.
- All three queries use the same eight contract names.
- Staging load disabled; only `Shipments` loaded.
- Profiling set to the entire data set.
- Rail date trap verified: MR-1003 = 03-Feb-2026 and MR-1012 = 13-Feb-2026.
- Final model: 34 rows, 8 columns, 578,650 kg.